# Dolphin-Yi-34B — GSS Demographic Simulation

Self-contained Colab notebook.  
Loads `cognitivecomputations/dolphin-2.9.1-yi-1.5-34b` from HuggingFace Hub,  
then runs the GSS demographic simulation across **3 prompt conditions × public + private topics**.

**Axes:**
- Respondents: GSS D/R respondents, 83-variable demographic profiles (10% stratified sample)
- Conditions: `rhetorical`, `stance`, `survey`
- Output: per-topic Mahalanobis distance (D vs R activations) correlated with GSS survey polarization

**Data folder:** upload `data/` alongside this notebook (or set `DATA_DIR` below).

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
import subprocess, sys
def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip('transformers>=4.40', 'accelerate', 'scipy', 'scikit-learn',
    'joblib', 'huggingface_hub', 'pandas', 'numpy')
print('Dependencies ready.')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
import os

MODEL_ID    = 'cognitivecomputations/dolphin-2.9.1-yi-1.5-34b'
MODEL_NAME  = 'dphn-yi34b'
SYSTEM_MSG  = 'You are simulating the views of an American.'

# Colab H100-96GB: Yi-34B in BF16 ~68 GB, ~28 GB headroom.
# Combined batch = all 3 conditions in one pass → effective per-condition
# batch is BATCH_SIZE // 3.  Increase to 48 if no OOM.
BATCH_SIZE  = 24      # prompts per GPU forward pass
MAX_LENGTH  = 512
PCA_DIM     = 15
SAMPLE_FRAC = 0.1
SAMPLE_SEED = 42

CONDITIONS  = ['rhetorical', 'stance', 'survey']

EXCLUDED_PUBLIC  = ['hubbywk1','racdif1','racdif2','racdif3','racdif4',
                    'workwhts','wlthwhts','intlwhts']
EXCLUDED_PRIVATE = ['reborn','marwht','helpful','helpfulnv','helpfulv']

# ── Paths ─────────────────────────────────────────────────────────────────────
# DATA_DIR: folder containing the 8 CSV/DTA files.
# Adjust if you placed data/ elsewhere (e.g. in Drive).
DATA_DIR   = '/content/data'
if not os.path.isdir(DATA_DIR):
    DATA_DIR = '/content/drive/MyDrive/Polarization/data'

GDRIVE_OUT = '/content/drive/MyDrive/Polarization'

GSS_CSV         = f'{DATA_DIR}/gss_2021_2024.csv'
STATA_2024      = f'{DATA_DIR}/GSS2024.dta'
STATA_2022      = f'{DATA_DIR}/GSS2022.dta'
DEMO_CSV        = f'{DATA_DIR}/gss_demographic_variables.csv'
PUBLIC_TOPICS   = f'{DATA_DIR}/public_issues.csv'
PUBLIC_POL      = f'{DATA_DIR}/public_issues_polarization.csv'
PRIVATE_TOPICS  = f'{DATA_DIR}/private_life.csv'
PRIVATE_POL     = f'{DATA_DIR}/private_life_polarization.csv'

print(f'DATA_DIR = {DATA_DIR}')
print(f'Files present: {[os.path.basename(p) for p in [GSS_CSV,STATA_2024,STATA_2022] if os.path.isfile(p)]}')

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(GDRIVE_OUT, exist_ok=True)
    print(f'Drive mounted. Results → {GDRIVE_OUT}')
except ImportError:
    GDRIVE_OUT = '/content/results'
    os.makedirs(GDRIVE_OUT, exist_ok=True)
    print(f'Not Colab — saving locally to {GDRIVE_OUT}')

In [ ]:

# ── Core: model loading + activation extraction ───────────────────────────────
# Inlined from model_utils.py with local_files_only removed for HF Hub download.

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM


def load_model_colab(model_id: str):
    """Load model from HF Hub (or local path) with auto dtype."""
    print(f'Loading {model_id} ...')
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    tokenizer.padding_side    = 'left'
    tokenizer.truncation_side = 'left'
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=dtype, device_map='auto', attn_implementation='sdpa'
    )
    model.generation_config.pad_token_id = tokenizer.pad_token_id

    cfg = model.config
    L   = cfg.num_hidden_layers
    H   = cfg.num_attention_heads
    D   = getattr(cfg, 'head_dim', cfg.hidden_size // H)
    print(f'  Loaded  layers={L}  Q-heads={H}  head_dim={D}  dtype={dtype}')
    return model, tokenizer


@torch.no_grad()
def extract_heads_batched(model, tokenizer, texts, system_msg,
                          batch_size=24, max_length=512):
    """
    Extract last-token Q-head activations for every attention layer.
    Returns (N, L, H, D) float32 numpy array.

    Hooks the INPUT to o_proj at each layer — captures the concatenated
    Q-head outputs before the output projection.
    """
    model.eval()
    cfg  = model.config
    L    = cfg.num_hidden_layers
    H    = cfg.num_attention_heads
    D    = getattr(cfg, 'head_dim', cfg.hidden_size // H)

    layer_outputs = [None] * L

    def get_hook(li):
        def hook(module, inp, out):
            # inp[0]: (batch, seq, H*D) → take last token → (B, H, D) on CPU
            x = inp[0].detach()
            layer_outputs[li] = (
                x[:, -1, :].float()
                 .view(x.shape[0], H, D)
                 .cpu().numpy()
            )
        return hook

    hooks = [model.model.layers[li].self_attn.o_proj
                 .register_forward_hook(get_hook(li))
             for li in range(L)]

    def fmt(text):
        if getattr(tokenizer, 'chat_template', None):
            try:
                return tokenizer.apply_chat_template(
                    [{'role': 'system', 'content': system_msg},
                     {'role': 'user',   'content': text}],
                    tokenize=False, add_generation_prompt=True)
            except Exception:
                pass
        return f'{system_msg}\n\n{text}'

    all_acts = []
    try:
        for i in range(0, len(texts), batch_size):
            batch = [fmt(t) for t in texts[i:i + batch_size]]
            enc   = tokenizer(batch, return_tensors='pt', padding=True,
                              truncation=True, max_length=max_length
                              ).to(model.device)
            model(**enc)
            del enc          # release GPU tensor immediately after forward pass
            all_acts.append(np.stack(layer_outputs, axis=1))
    finally:
        for h in hooks:
            h.remove()

    return np.concatenate(all_acts, axis=0)   # (N, L, H, D)


In [ ]:

# ── Core: PCA-Mahalanobis metrics ─────────────────────────────────────────────
# Inlined from run_gss_pca.py (only the functions we need).

import warnings
from scipy.linalg import inv, LinAlgError
from scipy.spatial.distance import mahalanobis
from sklearn.decomposition import PCA
from joblib import Parallel, delayed


def _wt_mean(X, w):
    w = w / w.sum()
    return (X * w[:, None]).sum(0)


def _wt_median(X, w):
    """Weighted median along axis 0 (component-wise)."""
    w = w / w.sum()
    result = np.zeros(X.shape[1])
    for j in range(X.shape[1]):
        order      = np.argsort(X[:, j])
        cumw       = np.cumsum(w[order])
        idx        = min(np.searchsorted(cumw, 0.5), len(X) - 1)
        result[j]  = X[order[idx], j]
    return result


def _wt_cov(X, w):
    w  = w / w.sum()
    mu = _wt_mean(X, w)
    Xc = X - mu
    C  = sum(w[i] * np.outer(Xc[i], Xc[i]) for i in range(len(X)))
    return C / (1 - (w**2).sum())


def _mahal_one_head(hd, labels, n_comp=15, centroid_method='mean'):
    """PCA-Mahalanobis for a single head's (N, D) activations."""
    mask = np.isin(labels, (100, 200))
    X, y = hd[mask], labels[mask]
    n    = min(n_comp, X.shape[1], X.shape[0] - 1)
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore')
            Xp = PCA(n_components=n).fit_transform(X)
        g1, g2 = Xp[y == 100], Xp[y == 200]
        if len(g1) < 6 or len(g2) < 6:
            return 0.0
        w1, w2   = np.ones(len(g1)), np.ones(len(g2))
        centroid = _wt_median if centroid_method == 'median' else _wt_mean
        mu1, mu2 = centroid(g1, w1), centroid(g2, w2)
        # Covariance always uses mean-based estimate (standard practice)
        C = (_wt_cov(g1, w1) + _wt_cov(g2, w2)) / 2
        C += np.eye(C.shape[0]) * 1e-6
        return float(mahalanobis(mu1, mu2, inv(C)))
    except (LinAlgError, ValueError):
        return 0.0


def compute_head_grid(X_heads, labels, n_comp=15, centroid_method='mean', n_jobs=-1):
    """
    (N, L, H, D) → (L, H) Mahalanobis grid, parallelised over heads.

    centroid_method: 'mean' or 'median' — controls how each party's
    centroid is computed in PCA space before taking the Mahalanobis distance.
    Covariance is always mean-based regardless.
    """
    N, L, H, D = X_heads.shape
    flat = [X_heads[:, l, h, :] for l in range(L) for h in range(H)]
    res  = Parallel(n_jobs=n_jobs)(
               delayed(_mahal_one_head)(hd, labels, n_comp, centroid_method)
               for hd in flat)
    return np.array(res, dtype=np.float32).reshape(L, H)


In [ ]:
# ── GSS data loading + demographic profile precomputation ─────────────────────
# Profile building is CPU-bound and embarrassingly parallel: each respondent
# is independent.  joblib.Parallel over 40+ Colab CPUs gives ~10x speedup.

import gc
import pandas as pd

DEMO_VARS = {
    'age','agekdbrn','babies','born','childs','degree','denom','denom16',
    'dipged','divorce','earnrs','educ','evwork','famdif16','family16',
    'granborn','health','hompop','hrs1','hrs2','incom16','income',
    'indus10','madeg','maeduc','maind10','major1','maocc10','marital',
    'mawrkgrw','mawrkslf','mobile16','numemps','occ10','othlang',
    'othlang1','othlang2','padeg','paeduc','paind10','paocc10','parborn',
    'partfull','pawrkslf','polviews','posslq','posslqy','preteen','race',
    'reg16','region','relig','relig16','res16','rincome','sex','sexornt',
    'sibs','spdeg','spden','speduc','spevwork','sphrs1','sphrs2',
    'spind10','spklang','spocc10','sprel','spwrkslf','spwrksta','teens',
    'unemp','unrelat','weekswrk','widowed','wksub','wksubs','wksup',
    'wksups','wrkslf','wrkstat','xnorcsiz','yousup',
}
STRATIFY_COLS = ['polviews','age_bin','degree','race','sex','rincome']


def build_code_maps(fields):
    print('  Building code maps from Stata files ...')
    df24n = pd.read_stata(STATA_2024, convert_categoricals=False)
    df24c = pd.read_stata(STATA_2024, convert_categoricals=True)
    df22n = pd.read_stata(STATA_2022, convert_categoricals=False)
    rdr22 = pd.io.stata.StataReader(STATA_2022)
    vl22  = rdr22.value_labels()
    v2l22 = dict(zip(rdr22._varlist, rdr22._lbllist))

    def _one(var):
        if var in df24n.columns:
            m = {int(n): str(c).strip()
                 for n, c in zip(df24n[var], df24c[var])
                 if pd.notna(n) and pd.notna(c) and int(n) < 100_000}
            if m: return m
        if var in df22n.columns:
            lbl = v2l22.get(var, '')
            if lbl and lbl in vl22:
                return {int(k): str(v).strip()
                        for k, v in vl22[lbl].items() if int(k) < 100_000}
        return {}

    maps = {f: _one(f) for f in fields}
    del df24n, df24c, df22n; gc.collect()
    return maps


def _build_one_respondent(idx, row_dict, code_maps, active_fields, all_labels):
    """Build (field, 'Label: value') tuples for one respondent."""
    parts = []
    for f in active_fields:
        val = row_dict.get(f)
        if val is None or (isinstance(val, float) and np.isnan(val)):
            continue
        code  = int(val)
        label = all_labels.get(f, f)
        text  = code_maps[f].get(code, str(code))
        parts.append((f, f'{label}: {text}'))
    return idx, parts


def load_gss_data():
    print('Loading GSS CSV ...')
    df = pd.read_csv(GSS_CSV, low_memory=False)
    print(f'  {df.shape[0]} rows, {df.shape[1]} cols')

    df['party_code'] = np.nan
    df.loc[df['partyid'].isin([0,1,2]), 'party_code'] = 100
    df.loc[df['partyid'].isin([4,5,6]), 'party_code'] = 200
    df_dr = df[df['party_code'].notna()].copy()
    print(f'  D/R: {len(df_dr)}  '
          f'(D={(df_dr.party_code==100).sum()}, R={(df_dr.party_code==200).sum()})')

    df_dr['age_bin'] = pd.cut(df_dr['age'], [0,30,40,50,60,70,100], labels=False)
    sc = [c for c in STRATIFY_COLS if c in df_dr.columns]
    df_dr['_strat'] = ''
    for c in sc:
        df_dr['_strat'] += df_dr[c].fillna(-1).astype(int).astype(str) + '_'

    # Demographic field labels
    demo_df  = pd.read_csv(DEMO_CSV)
    all_lbl  = dict(zip(demo_df['VariableName'], demo_df['ConciseDescription']))
    active   = sorted(DEMO_VARS)
    for f in active:
        all_lbl.setdefault(f, f)

    code_maps = build_code_maps(active)

    print(f'  Precomputing profiles for {len(df_dr)} respondents '
          f'(parallel, n_jobs=-1) ...')
    rows_as_dicts = df_dr[active].to_dict(orient='index')
    results = Parallel(n_jobs=-1, prefer='threads')(
        delayed(_build_one_respondent)(idx, row, code_maps, active, all_lbl)
        for idx, row in rows_as_dicts.items()
    )
    demo_parts = {idx: parts for idx, parts in results}
    print(f'  Done. {len(demo_parts)} profiles.')
    return df_dr, demo_parts, set(active)


df_dr, demo_parts, active_fields_set = load_gss_data()
print('GSS data ready.')

In [ ]:
# ── Topic loading ─────────────────────────────────────────────────────────────

def load_topics():
    ex_pub = {e.lower() for e in EXCLUDED_PUBLIC}
    ex_prv = {e.lower() for e in EXCLUDED_PRIVATE}

    def _load(topics_csv, pol_csv, excluded_set, cat):
        t_df  = pd.read_csv(topics_csv)
        p_df  = pd.read_csv(pol_csv)
        valid = set(p_df['variable'])
        pol   = dict(zip(p_df['variable'], p_df['polarization']))
        out   = {}
        for _, row in t_df.iterrows():
            var = str(row['Variable']).strip()
            if var.lower() in excluded_set or var not in valid:
                continue
            td = str(row.get('NaturalLanguageClause') or '').strip()
            qt = str(row.get('SurveyQuestion')        or '').strip()
            out[var] = {'topic_desc': td or qt,
                        'question_text': qt,
                        'category': cat,
                        'gss_polarization': pol[var]}
        return out

    pub = _load(PUBLIC_TOPICS,  PUBLIC_POL,  ex_pub, 'public')
    prv = _load(PRIVATE_TOPICS, PRIVATE_POL, ex_prv, 'private')
    print(f'Topics: {len(pub)} public, {len(prv)} private')
    return pub, prv


public_topics, private_topics = load_topics()

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
import gc, torch

os_env_set = False
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

model, tokenizer = load_model_colab(MODEL_ID)

# Verify VRAM after loading
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {used:.1f} / {total:.1f} GB  ({total-used:.1f} GB free)')

In [ ]:
# ── Helpers: sampling + prompt building ──────────────────────────────────────

def stratified_sample(df_valid, frac, seed):
    rng     = np.random.default_rng(seed)
    sampled = []
    for _, grp in df_valid.groupby('_strat'):
        n = max(1, int(np.ceil(len(grp) * frac)))
        n = min(n, len(grp))
        sampled.append(df_valid.loc[rng.choice(grp.index, n, replace=False)])
    return pd.concat(sampled).sort_index()


def is_valid(val):
    try: return not pd.isna(val) and int(float(val)) < 1_000_000
    except: return False


def build_prompt(profile, topic_desc, question_text, condition):
    prefix = f'Given the following background about a person:\n{profile}\n\n'
    if condition == 'rhetorical':
        return prefix + f'Generate a statement by this person on {topic_desc}.'
    if condition == 'stance':
        return prefix + f"What is this person's position on {topic_desc}?"
    if condition == 'survey':
        return (prefix +
                f'If asked in a national survey about {question_text}, '
                f'how would this person respond?')
    raise ValueError(condition)


def sample_and_build_prompts(topic_name, topic_info):
    """
    Sample respondents and build prompts for ALL 3 conditions at once.

    Returns:
        prompts_all : list of 3*N strings  [cond0×N, cond1×N, cond2×N]
        labels      : (N,) int array
        N           : number of sampled respondents
    """
    if topic_name not in df_dr.columns:
        return None

    valid_mask = df_dr[topic_name].apply(is_valid)
    df_v = df_dr[valid_mask]
    if len(df_v) < 20:
        return None

    df_s = stratified_sample(df_v, SAMPLE_FRAC, SAMPLE_SEED)
    n_d  = int((df_s.party_code == 100).sum())
    n_r  = int((df_s.party_code == 200).sum())
    if n_d < 5 or n_r < 5:
        return None

    labels  = df_s['party_code'].values.astype(int)
    N       = len(df_s)
    exclude = topic_name if topic_name in active_fields_set else None

    # Build one profile per respondent (condition-independent)
    rng = np.random.default_rng(hash(topic_name) % (2**32))
    profiles = []
    for idx in df_s.index:
        parts = [p[1] for p in demo_parts[idx]
                 if exclude is None or p[0] != exclude]
        order = rng.permutation(len(parts))
        profiles.append('. '.join(parts[i] for i in order) + '.')

    td, qt = topic_info['topic_desc'], topic_info['question_text']

    # Build 3×N prompts: all conditions concatenated
    prompts_all = [
        build_prompt(p, td, qt, cond)
        for cond in CONDITIONS
        for p in profiles
    ]

    return prompts_all, labels, N, n_d, n_r

print('Helpers ready.')

In [ ]:

# ── Main simulation loop ──────────────────────────────────────────────────────
# Strategy: for each topic, pass ALL 3 conditions in a single
# extract_heads_batched call (3N prompts total).  This keeps the GPU
# busy continuously and avoids 3 separate Python/CUDA sync points.
# After extraction, split (3N, L, H, D) → 3 × (N, L, H, D) by condition.
#
# For each condition slice we compute TWO Mahalanobis grids:
#   grid_mean   — party centroids = weighted mean  (standard)
#   grid_median — party centroids = weighted median (robust to outliers)
# Both grids are (L, H) float32 and small, so holding them briefly is fine.

import time
from datetime import datetime

timestamp   = datetime.now().strftime('%Y%m%d_%H%M%S')
all_results = []

topic_sets = [('public', public_topics), ('private', private_topics)]

for cat_name, topics in topic_sets:
    print(f'\n{"="*70}')
    print(f'  Category: {cat_name.upper()}  ({len(topics)} topics)')
    print(f'{"="*70}')
    cat_start = time.time()

    for t_idx, (topic_name, topic_info) in enumerate(topics.items()):
        t0 = time.time()
        try:
            res = sample_and_build_prompts(topic_name, topic_info)
            if res is None:
                continue
            prompts_all, labels, N, n_d, n_r = res

            # Single forward pass for all 3 conditions (3N prompts total)
            X_all = extract_heads_batched(
                model, tokenizer, prompts_all, SYSTEM_MSG,
                batch_size=BATCH_SIZE, max_length=MAX_LENGTH,
            )  # (3N, L, H, D)

            L_model = X_all.shape[1]
            mid_s   = int(L_model * 0.45)
            mid_e   = max(int(L_model * 0.55), mid_s + 1)

            # Split by condition, compute mean + median grids, free each slice
            cond_mahals = []
            for ci, cond in enumerate(CONDITIONS):
                X_cond = X_all[ci * N:(ci + 1) * N]   # (N, L, H, D) — numpy view

                # Mean-centroid grid (standard)
                grid_mean   = compute_head_grid(X_cond, labels, PCA_DIM,
                                                centroid_method='mean')
                # Median-centroid grid (robust; covariance is still mean-based)
                grid_median = compute_head_grid(X_cond, labels, PCA_DIM,
                                                centroid_method='median')
                del X_cond                             # drop view before next slice

                all_results.append({
                    'model':            MODEL_NAME,
                    'topic':            topic_name,
                    'category':         cat_name,
                    'condition':        cond,
                    'n_sampled':        N,
                    'n_dem':            n_d,
                    'n_rep':            n_r,
                    'mahal_all':        float(np.mean(grid_mean)),
                    'mahal_all_median': float(np.mean(grid_median)),
                    'mahal_mid10':      float(np.mean(grid_mean[mid_s:mid_e])),
                    'mahal_max':        float(np.max(grid_mean)),
                    'gss_polarization': topic_info['gss_polarization'],
                })
                cond_mahals.append(float(np.mean(grid_mean)))
                del grid_mean, grid_median

            del X_all
            gc.collect()
            torch.cuda.empty_cache()

            elapsed = time.time() - t0
            print(f'  [{t_idx+1:3d}/{len(topics)}] {topic_name:14s}  '
                  f'n={N}  D={n_d} R={n_r}  '
                  f'rhet={cond_mahals[0]:.3f}  '
                  f'stan={cond_mahals[1]:.3f}  '
                  f'surv={cond_mahals[2]:.3f}  '
                  f'({elapsed:.0f}s)', flush=True)

        except Exception as e:
            print(f'  ERROR {topic_name}: {e}', flush=True)
            gc.collect()
            torch.cuda.empty_cache()

        # Checkpoint every 20 topics
        if (t_idx + 1) % 20 == 0 and all_results:
            ckpt = f'{GDRIVE_OUT}/{MODEL_NAME}_{cat_name}_ckpt_{timestamp}.csv'
            pd.DataFrame(all_results).to_csv(ckpt, index=False)
            print(f'  -- checkpoint saved ({len(all_results)} rows so far)')

    print(f'  {cat_name} done in {(time.time()-cat_start)/60:.1f} min')

    if all_results:
        cat_csv = f'{GDRIVE_OUT}/{MODEL_NAME}_{cat_name}_{timestamp}.csv'
        pd.DataFrame(all_results).to_csv(cat_csv, index=False)
        print(f'  Saved: {cat_csv}')

print(f'\nSimulation complete.  Total rows: {len(all_results)}')


In [ ]:
# ── Save final CSV + correlation summary ──────────────────────────────────────
from scipy.stats import pearsonr, spearmanr

df = pd.DataFrame(all_results)

final_csv = f'{GDRIVE_OUT}/{MODEL_NAME}_all_{timestamp}.csv'
df.to_csv(final_csv, index=False)
print(f'Saved: {final_csv}  ({len(df)} rows)\n')

# ── Correlation table: mean vs median centroid ────────────────────────────────
# mahal_all        = mean-centroid Mahalanobis (standard)
# mahal_all_median = median-centroid Mahalanobis (robust to outlier respondents)
# Both are averaged over all (L, H) heads.  Higher r/ρ vs GSS = better signal.

hdr = (f'{"Category":8s}  {"Condition":12s}  '
       f'{"mean r":>7s}  {"mean ρ":>7s}  '
       f'{"med r":>7s}  {"med ρ":>7s}  {"n":>4s}')
print(hdr)
print('-' * len(hdr))

for cat in ['public', 'private']:
    for cond in CONDITIONS:
        sub = df[(df.category == cat) & (df.condition == cond)].dropna(
                  subset=['gss_polarization', 'mahal_all', 'mahal_all_median'])
        if len(sub) < 5:
            continue
        r_m,   _ = pearsonr( sub.mahal_all,        sub.gss_polarization)
        rho_m, _ = spearmanr(sub.mahal_all,        sub.gss_polarization)
        r_med, _ = pearsonr( sub.mahal_all_median, sub.gss_polarization)
        rho_e, _ = spearmanr(sub.mahal_all_median, sub.gss_polarization)
        # Highlight whichever variant wins on Pearson r
        flag = '★' if r_med > r_m else ' '
        print(f'{cat:8s}  {cond:12s}  '
              f'{r_m:7.3f}  {rho_m:7.3f}  '
              f'{r_med:7.3f}{flag} {rho_e:7.3f}  {len(sub):4d}')

print('\n★ = median centroid outperforms mean on Pearson r vs GSS polarization')
